# Imports

In [2]:
!pip install -q uv && uv pip install -q autogluon.tabular


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
error: Failed to install: psutil-7.1.3-cp37-abi3-win_amd64.whl (psutil==7.1.3)
  Caused by: failed to rename file from C:\Users\admin\Desktop\kaggle_competition\predicting_stellar_class\env\Lib\site-packages\psutil\.tmpitJO9p\_psutil_windows.pyd to C:\Users\admin\Desktop\kaggle_competition\predicting_stellar_class\env\Lib\site-packages\psutil\_psutil_windows.pyd: Access is denied. (os error 5)


In [4]:
import warnings
import numpy as np
import pandas as pd
# from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
FOLDS = 5
ID = 'id'
TARGET = 'class'

# Data Loading & Original Dataset Preparation

In [9]:
train = pd.read_csv("playground-series-s6e6\\train.csv")
test = pd.read_csv("playground-series-s6e6\\train.csv")
original = pd.read_csv("playground-series-s6e6\\externel_dataset\\star_classification_pred_spectral_type_galaxy_pop_org.csv") 

train_id = train[ID]
test_id = test[ID]

def get_spectral_type(g, r):
    return pd.cut(
        r - g, 
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

def get_galaxy_population(u, r):
    return pd.cut(
        u - r, 
        [-np.inf, 1.4, 2.2, np.inf],
        labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
    ).astype(str)

original['spectral_type'] = get_spectral_type(original['g'], original['r'])
original['galaxy_population'] = get_galaxy_population(original['u'], original['r'])

target_map = {'GALAXY': 'GALAXY', 'QSO': 'QSO', 'STAR': 'STAR'}
original[TARGET] = original[TARGET].map(target_map)
train[TARGET] = train[TARGET].map(target_map)

base_features = [col for col in train.columns if col not in [TARGET, ID]]
cat_cols_init = train.drop(columns=[ID, TARGET]).select_dtypes(include=['object']).columns.tolist()
num_cols_init = [c for c in train.drop(columns=[ID, TARGET]).select_dtypes(exclude=['object']).columns]

original_numeric_target = original[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
original_global_mean = original_numeric_target.mean()
original_global_median = original_numeric_target.median()

original_stats = {}
for col in base_features:
    if col in original.columns:
        orig_temp = original[[col]].copy()
        orig_temp[TARGET] = original_numeric_target
        if col in num_cols_init:
            orig_temp[col] = np.floor(orig_temp[col])
            
        stats = orig_temp.groupby(col)[TARGET].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
        original_stats[col] = stats

# Feature Engineering Pipeline Definition

In [10]:
color_pairs = [('u', 'g'), ('g', 'r'), ('r', 'i'), ('i', 'z'), ('u', 'z')]
important_combos = sorted([
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
    ('g_cat_', 'r_cat_'), 
])

category_map = {}

def feature_engineering(df, fit=False):
    df = df.copy()
    eps = 1e-6

    for col, stats_df in original_stats.items():
        if col in df.columns:
            df_key = df[[col]].copy()
            if col in num_cols_init:
                df_key[col] = np.floor(df_key[col])
                
            merged_stats = df_key.merge(stats_df, on=col, how='left').drop(columns=[col])
            df = pd.concat([df, merged_stats], axis=1)
            
            fill_values = {
                f"orig_{col}_mean": original_global_mean, f"orig_{col}_median": original_global_median,
                f"orig_{col}_std": 0.0, f"orig_{col}_skew": 0.0, f"orig_{col}_count": 0.0
            }
            df.fillna(value=fill_values, inplace=True)
            for s in ['mean', 'median', 'std', 'skew', 'count']:
                df[f"orig_{col}_{s}"] = df[f"orig_{col}_{s}"].astype('float32')

    if 'redshift' in df.columns:
        df['_is_star_redshift'] = (df['redshift'] <= 0).astype('int32')
        df['_dist_modulus_proxy'] = (5 * np.log10(np.abs(df['redshift']) + eps)).astype('float32')
        if 'r' in df.columns: df['_absolute_r_proxy'] = (df['r'] - df['_dist_modulus_proxy']).astype('float32')
        if 'g' in df.columns: df['_absolute_g_proxy'] = (df['g'] - df['_dist_modulus_proxy']).astype('float32')

    bands = ['u', 'g', 'r', 'i', 'z']
    if all(b in df.columns for b in bands):
        df['_ugriz_skew'] = df[bands].skew(axis=1).astype('float32')
        df['_ugriz_kurt'] = df[bands].kurt(axis=1).astype('float32')
        df['_ugriz_mean'] = df[bands].mean(axis=1).astype('float32')
        df['_ugriz_std'] = df[bands].std(axis=1).astype('float32')
        df['_ugriz_max_diff'] = (df[bands].max(axis=1) - df[bands].min(axis=1)).astype('float32')

    if 'alpha' in df.columns and 'delta' in df.columns:
        alpha_rad, delta_rad = np.radians(df['alpha']), np.radians(df['delta'])
        
        df['_coord_x'] = (np.cos(delta_rad) * np.cos(alpha_rad)).astype('float32')
        df['_coord_y'] = (np.cos(delta_rad) * np.sin(alpha_rad)).astype('float32')
        df['_coord_z'] = (np.sin(delta_rad)).astype('float32')
        
        ra_gp = np.radians(192.85948)
        dec_gp = np.radians(27.12825)
        l_cp = np.radians(122.93314)
        
        sin_b = np.sin(delta_rad) * np.sin(dec_gp) + np.cos(delta_rad) * np.cos(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_b'] = np.degrees(np.arcsin(np.clip(sin_b, -1.0, 1.0))).astype('float32')
        
        y_l = np.cos(delta_rad) * np.sin(alpha_rad - ra_gp)
        x_l = np.sin(delta_rad) * np.cos(dec_gp) - np.cos(delta_rad) * np.sin(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_l'] = np.degrees(l_cp - np.arctan2(y_l, x_l)).astype('float32') % 360.0
        
        df['_sin_alpha'] = np.sin(alpha_rad).astype('float32')
        df['_cos_alpha'] = np.cos(alpha_rad).astype('float32')
        df['_sin_delta'] = np.sin(delta_rad).astype('float32')
        df['_cos_delta'] = np.cos(delta_rad).astype('float32')
        
        df['_alpha_plus_delta'] = (df['alpha'] + df['delta']).astype('float32')
        df['_alpha_minus_delta'] = (df['alpha'] - df['delta']).astype('float32')
        df['_alpha_mod1'] = (df['alpha'] % 1).astype('float32')
        df['_delta_mod1'] = (df['delta'] % 1).astype('float32')
        df['_total_spatial_distance'] = np.sqrt(df['alpha']**2 + df['delta']**2).astype('float32')
    
    if 'redshift' in df.columns:
        df['_log_redshift'] = np.log1p(np.abs(df['redshift'])).astype('float32')
        df['_r_x_redshift'] = (df['r'] * df['redshift']).astype('float32')
        df['_g_/_redshift'] = (df['g'] / (df['redshift'] + eps)).astype('float32')
        df['_i_/_redshift'] = (df['i'] / (df['redshift'] + eps)).astype('float32')
        
    for a, b in color_pairs:
        if a in df.columns and b in df.columns:
            df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    if all(b in df.columns for b in ['g', 'r', 'i']):
        df['_balmer_break_proxy'] = ((df['g'] - df['r']) - (df['r'] - df['i'])).astype('float32')

    if all(b in df.columns for b in ['u', 'g', 'r', 'i', 'z']):
        df['g_r'] = df['g'] - df['r']
        df['r_i'] = df['r'] - df['i']
        df['u_g'] = df['u'] - df['g']
        df['i_z'] = df['i'] - df['z']
        
        df['_color_cross_ug_gr'] = (df['u_g'] * df['g_r']).astype('float32')
        df['_color_cross_gr_ri'] = (df['g_r'] * df['r_i']).astype('float32')
        
        df['stellar_locus_dist'] = np.sqrt((df['g_r'] - 0.52)**2 + (df['r_i'] - 0.25)**2).astype('float32')
        df['qso_locus_dist'] = np.sqrt((df['g_r'] - 0.24)**2 + (df['r_i'] - 0.15)**2).astype('float32')
        df['galaxy_locus_dist'] = np.sqrt((df['u_g'] - 1.50)**2 + (df['g_r'] - 0.70)**2).astype('float32')

    for col in cat_cols_init:
        if fit:
            df[col] = pd.Categorical(df[col])
            category_map[col] = df[col].dtype.categories
        else:
            df[col] = pd.Categorical(df[col], categories=category_map[col])

    for col in num_cols_init:
        cat_name = f"{col}_cat_"
        floored_values = np.floor(df[col])
        if fit:
            df[cat_name] = pd.Categorical(floored_values)
            category_map[col] = df[cat_name].dtype.categories
        else:
            df[cat_name] = pd.Categorical(floored_values, categories=category_map[col])

    for col, bins_list in {'delta': [100, 500]}.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)
                category_map[bin_name] = kb
            else:
                binned = category_map[bin_name].transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)

    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]: 
            combo_series = combo_series + '_' + df[col].astype(str)
        
        if fit:
            df[combo_name] = pd.Categorical(combo_series)
            category_map[combo_name] = df[combo_name].dtype.categories
        else:
            df[combo_name] = pd.Categorical(combo_series, categories=category_map[combo_name])

    return df

# Feature Engineering Execution 

In [11]:
X_train = train.drop(columns=[ID, TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[ID])

X_train = feature_engineering(X_train, fit=True)
X_test = feature_engineering(X_test, fit=False)

SKEW_THRESHOLD = 1.5
UNIQUENESS_THRESHOLD = 15
current_num_cols = X_train.select_dtypes(exclude=['category', 'object']).columns.tolist()

for col in current_num_cols:
    if X_train[col].nunique() <= UNIQUENESS_THRESHOLD: continue
    if abs(X_train[col].skew()) > SKEW_THRESHOLD:
        X_train[col] = (np.sign(X_train[col]) * np.log1p(np.abs(X_train[col]))).astype('float32')
        X_test[col] = (np.sign(X_test[col]) * np.log1p(np.abs(X_test[col]))).astype('float32')

for col in X_train.columns.tolist():
    if isinstance(X_train[col].dtype, pd.CategoricalDtype) or X_train[col].dtype == 'object' or X_train[col].nunique() <= UNIQUENESS_THRESHOLD:
        X_train[col], X_test[col] = pd.Categorical(X_train[col]), pd.Categorical(X_test[col])

constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
if constant_cols:
    X_train.drop(columns=constant_cols, inplace=True)
    X_test.drop(columns=constant_cols, inplace=True)

In [12]:
X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_cat_,r_cat_,i_cat_,z_cat_,redshift_cat_,delta_100_quantile_bin_,delta_500_quantile_bin_,alpha_cat__delta_cat__,g_cat__r_cat__,u_cat__z_cat__
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.342868,M,Red_Sequence,...,21.0,20.0,19.0,18.0,0.0,43,215,147.0_16.0,21.0_20.0,25.0_18.0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.146673,M,Red_Sequence,...,19.0,17.0,17.0,16.0,0.0,66,331,127.0_32.0,19.0_17.0,20.0_16.0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,1.341237,O/B,Blue_Cloud,...,21.0,21.0,20.0,20.0,2.0,71,358,179.0_35.0,21.0_21.0,21.0_20.0
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.429246,M,Red_Sequence,...,21.0,19.0,18.0,17.0,0.0,91,457,225.0_48.0,21.0_19.0,23.0_17.0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.441965,M,Red_Sequence,...,19.0,18.0,17.0,17.0,0.0,46,233,141.0_19.0,19.0_18.0,21.0_17.0


## Best LightGBM 

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb
SEED = 42
RUN_SKF = True
N_FOLDS = 5

X_train_, X_test_, y_train_, y_test_ = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

best_lgb_params = {
                    'n_estimators': 1841, 'learning_rate': 0.016121467631636684, 'num_leaves': 126, 
                    'max_depth': 12, 'min_child_samples': 7, 'subsample': 0.9137011411523339, 
                    'colsample_bytree': 0.6472035788104408, 'reg_alpha': 1.6143965568012781, 
                    'reg_lambda': 0.0457226857453295, 'min_split_gain': 0.520062046419683
                    }
best_lgb_params['class_weight']='balanced'
best_lgb_params['random_state']=42

if RUN_SKF:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []

    print(f"{'Fold':<6} {'Balanced Accuracy':^20}")
    print("-" * 30)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_, y_train_), 1):
        X_tr, X_val = X_train_.iloc[train_idx], X_train_.iloc[val_idx]
        y_tr, y_val = y_train_.iloc[train_idx], y_train_.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
        **best_lgb_params,
        verbose=-1
        )
        
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(y_val, y_pred)
        fold_scores.append(balanced_acc)
        
        print(f"{fold:<6} {balanced_acc:>20.4f}")

    print("-" * 30)
    print(f"{'Mean':<6} {np.mean(fold_scores):>20.4f}")
    print(f"{'Std':<6} {np.std(fold_scores):>20.4f}")

Fold    Balanced Accuracy  
------------------------------
1                    0.9630
2                    0.9625
3                    0.9603
4                    0.9620


KeyboardInterrupt: 

# AutoGluon Training

In [ ]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
weight_map = dict(zip(classes, class_weights))

X_train['_sample_weight'] = y_train.map(weight_map).astype('float32')

X_train[TARGET] = y_train

predictor = TabularPredictor(
    label=TARGET, 
    eval_metric='balanced_accuracy', 
    path='AutogluonModels',
    sample_weight='_sample_weight',
)

predictor.fit(
    train_data=X_train,
    num_bag_folds=FOLDS,          
    num_bag_sets=1,
    auto_stack=True,              
    time_limit=3600*10,              
    presets='best_quality',
    ag_args_fit={"num_gpus": 1},
)

# predictor.fit(
#     train_data=X_train,
#     presets='best_quality',
#     time_limit=60,
#     num_bag_folds=0,
#     auto_stack=False
# )

# predictor.fit(
#     train_data=X_train,
#     presets='best_quality',
#     time_limit=300,
#     num_bag_folds=5,
#     auto_stack=False
# )

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.10.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 15.89/15.89 GB
Total GPU Memory:   Free: 15.89 GB, Allocated: 0.00 GB, Total: 15.89 GB
GPU Count:          1
Memory Avail:       24.41 GB / 31.35 GB (77.9%)
Disk Space Avail:   17.94 GB / 19.52 GB (91.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked

# Leaderboard 

In [ ]:
predictor.leaderboard(silent=True).style.background_gradient(subset=['score_val'], cmap='RdYlGn') 

# Output File Generation

In [ ]:
X_train_predict = X_train.drop(columns=[TARGET, '_sample_weight'])

oof_preds_prob = predictor.predict_proba(X_train_predict)
oof_df = pd.DataFrame({ID: train_id})
for cls in ['GALAXY', 'QSO', 'STAR']:
    oof_df[cls] = oof_preds_prob[cls]
oof_df.to_csv('oof_preds.csv', index=False)

test_preds_prob = predictor.predict_proba(X_test)
test_df = pd.DataFrame({ID: test_id})
for cls in ['GALAXY', 'QSO', 'STAR']:
    test_df[cls] = test_preds_prob[cls]
test_df.to_csv('test_preds.csv', index=False)

sub_preds = predictor.predict(X_test)
sub = pd.DataFrame({ID: test_id, TARGET: sub_preds})
sub.to_csv('submission_1_gpu.csv', index=False)
sub.head() 